<a href="https://colab.research.google.com/github/Aditya-Kumar-Sharma/IIT-Gandhinagar_PGD_Week-0_Assignment-1/blob/main/IITGN_Cohort2_Python_Assignment_26271001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aditya Kumar Sharma**

**Id - 26271001**

---




**Question 1**

*Clean Training Records Before Dataset Preparation*

In [10]:
def clean_training_records(records):
    cleaned = []
    rejected_indices = []

    for k, v in enumerate(records):
        learner_id = v.get("learner_id")
        score = v.get("score")

        if not isinstance(learner_id, str):
            rejected_indices.append(k)
            continue

        learner_id = learner_id.strip()

        if learner_id == "":
            rejected_indices.append(k)
            continue

        if isinstance(score, bool):
            rejected_indices.append(k)
            continue

        if isinstance(score, complex):
            rejected_indices.append(k)
            continue

        try:
            score = float(score)
        except (ValueError, TypeError):
            rejected_indices.append(k)
            continue

        if score > 100 or score < 0:
            rejected_indices.append(k)
            continue

        cleaned.append({
            "learner_id": learner_id,
            "score": score,
            "eligible": score >= 60
        })

    return cleaned, rejected_indices

In [11]:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

records

[{'learner_id': ' L01 ', 'score': '82.5'},
 {'learner_id': 'L02', 'score': 60},
 {'learner_id': 'L03', 'score': None},
 {'learner_id': '   ', 'score': 75},
 {'learner_id': 'L04', 'score': True},
 {'learner_id': 'L05', 'score': 48.0},
 {'learner_id': 'L06', 'score': 105}]

In [12]:
accepted, rejected = clean_training_records(records)

print("cleaned :")
for v in accepted:
    print(v)

print("\nRejected indices:")
print(rejected)

cleaned :
{'learner_id': 'L01', 'score': 82.5, 'eligible': True}
{'learner_id': 'L02', 'score': 60.0, 'eligible': True}
{'learner_id': 'L05', 'score': 48.0, 'eligible': False}

Rejected indices:
[2, 3, 4, 6]


In [13]:
records1 = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L05", "score": -5},
    {"learner_id": "L06", "score": 105},
]

accepted, rejected = clean_training_records(records1)

print("cleaned :")
for v in accepted:
    print(v)

print("\nRejected indices:")
print(rejected)

cleaned :
{'learner_id': 'L01', 'score': 82.5, 'eligible': True}

Rejected indices:
[1, 2]


In [14]:
records2 = [
    {"learner_id": "L04", "score": '5+2i'},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

accepted, rejected = clean_training_records(records2)

print("cleaned :")
for v in accepted:
    print(v)

print("\nRejected indices:")
print(rejected)

cleaned :
{'learner_id': 'L05', 'score': 48.0, 'eligible': False}

Rejected indices:
[0, 2]


In [15]:
records3 = [
    {"learner_id": "L04", "score": True},
    {"learner_id": 5, "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

accepted, rejected = clean_training_records(records3)

print("cleaned :")
for v in accepted:
    print(v)

print("\nRejected indices:")
print(rejected)

cleaned :

Rejected indices:
[0, 1, 2]


step 1 : drop blank learner_id
step 2 : drop none, 100 above, below 0 or complex number scores **using comprehensions**

step 3:add key value pair of eligibility with score above 60 eligible true

step 5 : compare and tell the droped indices



---



---



---



**Question 2**

*Prepare Mixed Text Payloads for an NLP Pipeline*

In [16]:
def prepare_messages(payloads, min_length=3):
    if isinstance(min_length, bool) or not isinstance(min_length, int):
        raise ValueError("min_length must be a positive integer")

    if min_length <= 0:
        raise ValueError("min_length must be a positive integer")

    token_lists = []
    rejected_indices = []

    punctuation = ".,!?;:"

    for index, payload in enumerate(payloads):
        if isinstance(payload, str):
            text = payload
        elif isinstance(payload, (bytes, bytearray)):
            try:
                text = bytes(payload).decode("utf-8")
            except UnicodeDecodeError:
                rejected_indices.append(index)
                continue
        else:
            rejected_indices.append(index)
            continue

        raw_tokens = text.lower().split()

        tokens = [
            token.strip(punctuation)
            for token in raw_tokens
        ]

        tokens = [
            token
            for token in tokens
            if len(token) >= min_length
        ]

        token_lists.append(tokens)

    vocabulary = sorted({
        token
        for tokens in token_lists
        for token in tokens
    })

    return {
        "token_lists": token_lists,
        "vocabulary": vocabulary,
        "rejected_indices": rejected_indices
    }

In [17]:
payloads = [
    " AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",
]

print(prepare_messages(payloads, min_length=3))

{'token_lists': [['helps', 'teams'], ['build', 'reliable', 'agents'], ['helps']], 'vocabulary': ['agents', 'build', 'helps', 'reliable', 'teams'], 'rejected_indices': [3, 4]}


**Question 3**


---



---



In [18]:
def summarise_predictions(results):
    valid_count = 0
    correct_count = 0
    by_actual = {}
    misclassified_ids = []
    rejected_indices = []

    for index, record in enumerate(results):
        if not isinstance(record, dict):
            rejected_indices.append(index)
            continue

        required_keys = {"id", "actual", "predicted"}

        if not required_keys.issubset(record):
            rejected_indices.append(index)
            continue

        if not all(isinstance(record[key], str) for key in required_keys):
            rejected_indices.append(index)
            continue

        record_id = record["id"].strip()
        actual = record["actual"].strip().lower()
        predicted = record["predicted"].strip().lower()

        if not record_id or not actual or not predicted:
            rejected_indices.append(index)
            continue

        valid_count += 1

        if actual not in by_actual:
            by_actual[actual] = {"total": 0, "correct": 0}

        by_actual[actual]["total"] += 1

        if actual == predicted:
            correct_count += 1
            by_actual[actual]["correct"] += 1
        else:
            misclassified_ids.append(record_id)

    accuracy = (
        round(correct_count / valid_count, 4)
        if valid_count
        else None
    )

    return {
        "valid_count": valid_count,
        "correct_count": correct_count,
        "accuracy": accuracy,
        "by_actual": by_actual,
        "misclassified_ids": misclassified_ids,
        "rejected_indices": rejected_indices
    }

In [19]:
results = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]

print(summarise_predictions(results))

{'valid_count': 4, 'correct_count': 3, 'accuracy': 0.75, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'misclassified_ids': ['T2'], 'rejected_indices': [4]}


**Question 4**


---



---

In [20]:
def audit_dataset_ids(train_ids, validation_ids):
    def process_ids(ids):
        valid_ids = []
        invalid_indices = []
        counts = {}

        for index, identifier in enumerate(ids):
            if not isinstance(identifier, str):
                invalid_indices.append(index)
                continue

            identifier = identifier.strip()

            if not identifier:
                invalid_indices.append(index)
                continue

            valid_ids.append(identifier)
            counts[identifier] = counts.get(identifier, 0) + 1

        unique_ids = set(valid_ids)
        duplicates = {
            identifier
            for identifier, count in counts.items()
            if count > 1
        }

        return unique_ids, duplicates, invalid_indices

    train_set, train_duplicates, invalid_train_indices = process_ids(train_ids)
    validation_set, validation_duplicates, invalid_validation_indices = process_ids(
        validation_ids
    )

    return {
        "train_duplicates": sorted(train_duplicates),
        "validation_duplicates": sorted(validation_duplicates),
        "overlap": sorted(train_set & validation_set),
        "train_only": sorted(train_set - validation_set),
        "validation_only": sorted(validation_set - train_set),
        "invalid_train_indices": invalid_train_indices,
        "invalid_validation_indices": invalid_validation_indices,
        "snapshots": {
            "train": frozenset(train_set),
            "validation": frozenset(validation_set),
        }
    }

In [21]:
train_ids = ["R3", "R1", " R2 ", "R1", None]
validation_ids = ["R2", "R4", "R4", " "]

print(audit_dataset_ids(train_ids, validation_ids))

{'train_duplicates': ['R1'], 'validation_duplicates': ['R4'], 'overlap': ['R2'], 'train_only': ['R1', 'R3'], 'validation_only': ['R4'], 'invalid_train_indices': [4], 'invalid_validation_indices': [3], 'snapshots': {'train': frozenset({'R2', 'R3', 'R1'}), 'validation': frozenset({'R2', 'R4'})}}


**Question 5**


---



---

In [22]:
def update_annotation_queue(
    queue,
    incoming,
    completed,
    priority=None,
    batch_size=3
):
    if isinstance(batch_size, bool) or not isinstance(batch_size, int):
        raise ValueError("batch_size must be a non-negative integer")

    if batch_size < 0:
        raise ValueError("batch_size must be a non-negative integer")

    combined = []
    seen = set()

    for document_id in queue + incoming:
        if document_id not in seen:
            combined.append(document_id)
            seen.add(document_id)

    completed_ids = set(completed)
    updated_queue = [
        document_id
        for document_id in combined
        if document_id not in completed_ids
    ]

    if priority in updated_queue:
        updated_queue.remove(priority)
        updated_queue.insert(0, priority)

    next_batch = updated_queue[:batch_size]
    last_item = updated_queue[-1] if updated_queue else None

    positions = [
        (position, updated_queue[position - 1])
        for position in range(1, len(updated_queue) + 1)
    ]

    return {
        "queue": updated_queue,
        "next_batch": next_batch,
        "last_item": last_item,
        "positions": positions
    }

In [23]:
queue = ["D3", "D1", "D3", "D2"]
incoming = ["D4", "D2", "D5"]
completed = ["D1", "D9"]

print(update_annotation_queue(
    queue,
    incoming,
    completed,
    priority="D5",
    batch_size=3
))

{'queue': ['D5', 'D3', 'D2', 'D4'], 'next_batch': ['D5', 'D3', 'D2'], 'last_item': 'D4', 'positions': [(1, 'D5'), (2, 'D3'), (3, 'D2'), (4, 'D4')]}


In [24]:
from copy import deepcopy
from typing import Any


def build_experiment_config(
    base: dict[str, Any],
    *,
    run_name: str,
    parameter_updates: dict[str, Any] | None = None,
    **metadata: Any
) -> dict[str, Any]:
    if not isinstance(run_name, str):
        raise ValueError("run_name must be a non-empty string")

    trimmed_run_name = run_name.strip()

    if not trimmed_run_name:
        raise ValueError("run_name must be a non-empty string")

    if parameter_updates is not None and not isinstance(parameter_updates, dict):
        raise TypeError("parameter_updates must be a dictionary or None")

    result = deepcopy(base)

    parameters = result["parameters"]

    if parameter_updates is not None:
        for key, value in parameter_updates.items():
            if key not in parameters:
                raise KeyError(key)
            parameters[key] = value

    result["metadata"].update(metadata)
    result["run_name"] = trimmed_run_name

    return result

In [25]:
base = {
    "model": "ticket_classifier",
    "parameters": {
        "threshold": 0.5,
        "max_tokens": 128
    },
    "metadata": {
        "owner": "platform",
        "stage": "baseline"
    }
}

print(build_experiment_config(
    base,
    run_name="trial_02",
    parameter_updates={"threshold": 0.7},
    owner="nlp_team",
    seed=42
))

{'model': 'ticket_classifier', 'parameters': {'threshold': 0.7, 'max_tokens': 128}, 'metadata': {'owner': 'nlp_team', 'stage': 'baseline', 'seed': 42}, 'run_name': 'trial_02'}


**Question 6**


---



---

In [26]:
import math


def aggregate_scores(*batches, top_n=3):
    if isinstance(top_n, bool) or not isinstance(top_n, int):
        raise ValueError("top_n must be a non-negative integer")

    if top_n < 0:
        raise ValueError("top_n must be a non-negative integer")

    valid_scores = []
    rejected_positions = []

    for batch_index, batch in enumerate(batches):
        if not isinstance(batch, (list, tuple)):
            raise TypeError("Each batch must be a list or tuple")

        for item_index, score in enumerate(batch):
            if (
                isinstance(score, bool)
                or not isinstance(score, (int, float))
                or not math.isfinite(float(score))
                or not 0 <= score <= 100
            ):
                rejected_positions.append((batch_index, item_index))
                continue

            valid_scores.append(
                (batch_index, item_index, float(score))
            )

    valid_count = len(valid_scores)

    if valid_count:
        mean = round(
            sum(score for _, _, score in valid_scores) / valid_count,
            2
        )
    else:
        mean = None

    ranked_scores = sorted(
        valid_scores,
        key=lambda item: (-item[2], item[0], item[1])
    )

    return {
        "valid_count": valid_count,
        "mean": mean,
        "top_scores": ranked_scores[:top_n],
        "rejected_positions": rejected_positions
    }

In [27]:
batch_a = [80, 90, None]
batch_b = (90, 70, True)
batch_c = []

print(aggregate_scores(batch_a, batch_b, batch_c, top_n=3))

{'valid_count': 4, 'mean': 82.5, 'top_scores': [(0, 1, 90.0), (1, 0, 90.0), (0, 0, 80.0)], 'rejected_positions': [(0, 2), (1, 2)]}


**Question 7**


---



---

In [28]:
def summarise_tool_calls(records):
    final_records = {}

    for record in records:
        final_records[record["call_id"]] = record.copy()

    tool_names = {
        record["tool"]
        for record in final_records.values()
    }

    by_tool = {
        tool: {
            "calls": 0,
            "successes": 0,
            "failures": 0,
            "total_duration_ms": 0
        }
        for tool in tool_names
    }

    failed_call_ids = []

    for call_id, record in final_records.items():
        tool = record["tool"]
        status = record["status"]
        duration = record["duration_ms"]

        summary = by_tool[tool]
        summary["calls"] += 1
        summary["total_duration_ms"] += duration

        if status == "ok":
            summary["successes"] += 1
        else:
            summary["failures"] += 1
            failed_call_ids.append(call_id)

    tool_ranking = sorted(
        by_tool,
        key=lambda tool: (
            -by_tool[tool]["failures"],
            -by_tool[tool]["calls"],
            tool
        )
    )

    return {
        "unique_calls": len(final_records),
        "by_tool": by_tool,
        "failed_call_ids": sorted(failed_call_ids),
        "tool_ranking": tool_ranking
    }

In [29]:
records = [
    {
        "call_id": "C1",
        "tool": "search",
        "status": "error",
        "duration_ms": 120
    },
    {
        "call_id": "C2",
        "tool": "calculator",
        "status": "ok",
        "duration_ms": 10
    },
    {
        "call_id": "C1",
        "tool": "search",
        "status": "ok",
        "duration_ms": 80
    },
    {
        "call_id": "C3",
        "tool": "search",
        "status": "error",
        "duration_ms": 40
    },
    {
        "call_id": "C4",
        "tool": "retriever",
        "status": "error",
        "duration_ms": 50
    },
]

print(summarise_tool_calls(records))

{'unique_calls': 4, 'by_tool': {'retriever': {'calls': 1, 'successes': 0, 'failures': 1, 'total_duration_ms': 50}, 'search': {'calls': 2, 'successes': 1, 'failures': 1, 'total_duration_ms': 120}, 'calculator': {'calls': 1, 'successes': 1, 'failures': 0, 'total_duration_ms': 10}}, 'failed_call_ids': ['C3', 'C4'], 'tool_ranking': ['search', 'retriever', 'calculator']}


**Question 8**


---



---

In [30]:
from functools import wraps
from typing import Callable, Any


def validate_text_batch(function: Callable) -> Callable:
    @wraps(function)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        if args:
            texts = args[0]
        elif "texts" in kwargs:
            texts = kwargs["texts"]
        else:
            raise TypeError("texts argument is required")

        if not isinstance(texts, list):
            raise TypeError("texts must be a list")

        if not all(isinstance(text, str) for text in texts):
            raise TypeError("every item must be a string")

        if any(not text.strip() for text in texts):
            raise ValueError("blank strings are not allowed")

        return function(*args, **kwargs)

    return wrapper


@validate_text_batch
def text_lengths(texts: list[str], *, strip: bool = True) -> list[int]:
    """Return the character length of each text."""
    if strip:
        return [len(text.strip()) for text in texts]

    return [len(text) for text in texts]

In [31]:
texts = [" agent ", "ML", "data"]

print(text_lengths(texts))
print(text_lengths(texts=texts, strip=False))

[5, 2, 4]
[7, 2, 4]


**Question 9**


---



---

In [32]:
import math
from functools import wraps


def track_calls(history):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            call_number = len(history) + 1

            try:
                result = function(*args, **kwargs)
            except Exception as error:
                history.append({
                    "call_number": call_number,
                    "function": function.__name__,
                    "status": "error",
                    "error_type": type(error).__name__
                })
                raise
            else:
                history.append({
                    "call_number": call_number,
                    "function": function.__name__,
                    "status": "success",
                    "error_type": None
                })
                return result

        return wrapper

    return decorator


history = []


@track_calls(history)
def average_score(scores):
    if not isinstance(scores, list):
        raise TypeError("scores must be a list")

    if not scores:
        raise ValueError("scores cannot be empty")

    for score in scores:
        if (
            isinstance(score, bool)
            or not isinstance(score, (int, float))
            or not math.isfinite(float(score))
        ):
            raise ValueError("scores must contain finite numbers")

    return sum(scores) / len(scores)


@track_calls(history)
def format_run_name(prefix: str, *, number: int) -> str:
    if not isinstance(prefix, str) or not prefix:
        raise ValueError("prefix must be non-empty")

    if (
        isinstance(number, bool)
        or not isinstance(number, int)
        or number < 0
    ):
        raise ValueError("number must be a non-negative integer")

    return f"{prefix}_{number:02d}"

In [33]:
first_result = average_score([60, 80])
second_result = format_run_name("eval", number=3)

try:
    average_score([])
except ValueError:
    pass

print(first_result)
print(second_result)
print(history)

70.0
eval_03
[{'call_number': 1, 'function': 'average_score', 'status': 'success', 'error_type': None}, {'call_number': 2, 'function': 'format_run_name', 'status': 'success', 'error_type': None}, {'call_number': 3, 'function': 'average_score', 'status': 'error', 'error_type': 'ValueError'}]


In [34]:
assert average_score([0, 100]) == 50.0
assert format_run_name("run", number=0) == "run_00"
assert format_run_name("run", number=123) == "run_123"

separate_history = []


@track_calls(separate_history)
def returns_none():
    return None


assert returns_none() is None
assert separate_history == [{
    "call_number": 1,
    "function": "returns_none",
    "status": "success",
    "error_type": None
}]

try:
    average_score([True, 50])
except ValueError:
    pass
else:
    raise AssertionError("Boolean scores should be rejected")